# CICIDS-2017 — Exploratory Data Analysis (EDA) & Visualizations

This notebook performs comprehensive EDA on the preprocessed CICIDS-2017 dataset.
Run `DV_project_Preprocessing.ipynb` first to generate `modified_daata.csv`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

DATA_PATH  = r'C:/Users/HP/OneDrive/Desktop/Data-Analysis-for-Network-Traffic-Analysis-main/Data/modified_daata.csv'
IMAGES_DIR = r'C:/Users/HP/OneDrive/Desktop/Data-Analysis-for-Network-Traffic-Analysis-main/images'

import os; os.makedirs(IMAGES_DIR, exist_ok=True)
print('Ready. matplotlib:', matplotlib.__version__)

Ready. matplotlib: 3.11.2


## 1. Load & Clean Data

In [2]:
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
print('Shape:', df.shape)
print('Label distribution:')
print(df['Label'].value_counts())
df.head(3)

Shape: (3198475, 79)
Label distribution:
Label
BENIGN              2473419
PortScan             317608
DoS Hulk             230124
DDoS                 128025
FTP-Patator           15870
SSH-Patator           11794
DoS GoldenEye         10293
DoS slowloris          5796
DoS Slowhttptest       5499
Infiltration             36
Heartbleed               11
Name: count, dtype: int64


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,88,640,7,4,440,358,220,0,62.857143,107.349008,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,88,900,9,4,600,2944,300,0,66.666667,132.287566,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,88,1205,7,4,2776,2830,1388,0,396.571429,677.274651,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [3]:
null_cols = [c for c in df.columns if df[c].isnull().any()]
print('Null columns:', null_cols if null_cols else 'None — all clean')
print('Data types:\n', df.dtypes.value_counts())

Null columns: None — all clean
Data types:
 int64      54
float64    24
object      1
Name: count, dtype: int64


## 2. Distribution of Attack Types

In [4]:
val_counts = df['Label'].value_counts().reset_index()
val_counts.columns = ['Label', 'Count']

plt.figure(figsize=(14, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(val_counts)))
bars = plt.bar(val_counts['Label'], val_counts['Count'], color=colors, edgecolor='black', lw=0.5)
for bar, v in zip(bars, val_counts['Count']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
             f'{v:,}', ha='center', va='bottom', fontsize=8, rotation=45)
plt.xlabel('Attack / Traffic Type', fontsize=12)
plt.ylabel('Number of Flows (log scale)', fontsize=12)
plt.title('Distribution of Attack Type in CICIDS 2017 Dataset', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yscale('log')
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/Distribution of Attack Type in CICIDS 2017 Dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print('Note: Log scale used due to high class imbalance.')

Note: Log scale used due to high class imbalance.


## 3. Column Type Distribution (Pie)

In [5]:
n_numeric = len(df.select_dtypes(include=['int64', 'float64']).columns)
n_categ   = len(df.select_dtypes(include=['object']).columns)
plt.figure(figsize=(7, 7))
plt.pie([n_numeric, n_categ],
        labels=[f'Numerical ({n_numeric})', f'Categorical ({n_categ})'],
        autopct='%1.1f%%', startangle=90,
        colors=['#4472C4', '#ED7D31'], explode=(0.05, 0))
plt.title('Distribution of Column Types', fontsize=13)
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/Distribution of Column Types.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Protocol Analysis

In [6]:
if 'Protocol' in df.columns:
    proto_counts = df['Protocol'].value_counts()
    proto_label_counts = df.groupby(['Protocol', 'Label']).size().reset_index(name='Count')

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    bars = axes[0].bar(proto_counts.index.astype(str), proto_counts.values,
                       color='steelblue', edgecolor='black', lw=0.5)
    for bar, v in zip(bars, proto_counts.values):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                     f'{v:,}', ha='center', va='bottom', fontsize=9)
    axes[0].set_title('Count of Protocol', fontsize=13)
    axes[0].set_xlabel('Protocol'); axes[0].set_ylabel('Count')

    pivot = proto_label_counts.pivot_table(index='Label', columns='Protocol', values='Count', fill_value=0)
    sns.heatmap(pivot, annot=True, fmt=',d', cmap='YlOrRd', ax=axes[1])
    axes[1].set_title('Count of Label across various Protocols', fontsize=13)
    axes[1].set_xlabel('Protocol'); axes[1].set_ylabel('Label')

    plt.tight_layout()
    plt.savefig(f'{IMAGES_DIR}/Count of Protocol.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Protocol column not found — skipping.')

Protocol column not found — skipping.


## 5. Flow Duration Analysis

In [7]:
sample_df = df.sample(min(50000, len(df)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 5.1 Box plot — fix: use tick_labels (mpl >= 3.9) or set manually
unique_labels = sorted(sample_df['Label'].unique())
duration_by_label = [sample_df[sample_df['Label'] == lbl]['Flow Duration'].values
                     for lbl in unique_labels]

bp = axes[0].boxplot(duration_by_label, patch_artist=True, showfliers=False)
# Set tick labels manually (works across all matplotlib versions)
axes[0].set_xticks(range(1, len(unique_labels) + 1))
axes[0].set_xticklabels(unique_labels, rotation=45, ha='right', fontsize=8)
colors_bp = plt.cm.Set3(np.linspace(0, 1, len(unique_labels)))
for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color)
axes[0].set_title('Box Plot of Flow Duration for Various Attack Types', fontsize=12)
axes[0].set_xlabel('Attack Type'); axes[0].set_ylabel('Flow Duration (microseconds)')

# 5.2 Average bar chart
avg_duration = df.groupby('Label')['Flow Duration'].mean().sort_values()
avg_duration.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='black', lw=0.5)
axes[1].set_title('Average Flow Duration by Attack Type', fontsize=12)
axes[1].set_xlabel('Mean Flow Duration (microseconds)')
axes[1].set_ylabel('Label')

plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/Box Plot of Flow Duration for Various Attack Types.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Average Flow Duration per Attack Type

In [8]:
avg_per_label = df.groupby('Label')['Flow Duration'].mean().sort_values(ascending=False)

plt.figure(figsize=(13, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(avg_per_label)))
bars = plt.bar(avg_per_label.index, avg_per_label.values, color=colors, edgecolor='black', lw=0.5)
for bar, v in zip(bars, avg_per_label.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f'{v:.0f}', ha='center', va='bottom', fontsize=8)
plt.title('Average Flow Duration for Each Attack Type', fontsize=13)
plt.xlabel('Attack Type'); plt.ylabel('Mean Flow Duration (µs)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/Average Flow Duration for Each Day.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Flag Count Analysis by Label

In [9]:
flag_cols = [c for c in df.columns if 'Flag' in c]
print('Flag columns found:', flag_cols)

if flag_cols:
    avg_flags = df.groupby('Label')[flag_cols].mean()
    fig, ax = plt.subplots(figsize=(14, 7))
    avg_flags.T.plot(kind='bar', ax=ax, edgecolor='black', lw=0.3)
    ax.set_title('Average Flag Counts by Label', fontsize=13)
    ax.set_xlabel('Flag Type'); ax.set_ylabel('Mean Count')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
    ax.tick_params(axis='x', rotation=0)
    plt.tight_layout()
    plt.savefig(f'{IMAGES_DIR}/Average of Various Flag Count for each Label.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No flag columns found.')

Flag columns found: ['Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count']


## 8. Correlation Analysis

In [10]:
numeric_df = df.select_dtypes(include=['number'])

# Top 25 highest-variance features for readability
top_var_cols = numeric_df.var().sort_values(ascending=False).head(25).index.tolist()
corr_top = numeric_df[top_var_cols].corr()

plt.figure(figsize=(18, 14))
mask = np.triu(np.ones_like(corr_top, dtype=bool))  # upper triangle mask
sns.heatmap(corr_top, mask=mask, annot=True, cmap='coolwarm', fmt='.2f',
            linewidths=0.3, vmin=-1, vmax=1, center=0,
            annot_kws={'size': 7})
plt.title('Correlation Heatmap — Top 25 High-Variance Features', fontsize=14)
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/Correlation Heatmap between attributes.png', dpi=150, bbox_inches='tight')
plt.show()

In [11]:
# Correlation of each feature with binary label (Attack=1, Benign=0)
df_corr = numeric_df.copy()
df_corr['Is_Attack'] = (df['Label'] != 'BENIGN').astype(int)
label_corr = df_corr.corr()['Is_Attack'].drop('Is_Attack').sort_values(key=abs, ascending=False)

plt.figure(figsize=(12, 8))
colors_corr = ['crimson' if v > 0 else 'steelblue' for v in label_corr.values[:25]]
label_corr.head(25).plot(kind='barh', color=colors_corr, edgecolor='black', lw=0.3)
plt.axvline(0, color='black', lw=1)
plt.title('Top 25 Features — Correlation with Attack Label', fontsize=13)
plt.xlabel('Pearson Correlation with Is_Attack')
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/Correlation Matrix between Attributes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top positive (attack indicators):', label_corr[label_corr > 0].head(5).index.tolist())
print('Top negative (benign indicators):', label_corr[label_corr < 0].head(5).index.tolist())

Top positive (attack indicators): ['Bwd Packet Length Std', 'Bwd Packet Length Max', 'Packet Length Variance', 'Bwd Packet Length Mean', 'Avg Bwd Segment Size']
Top negative (benign indicators): ['Min Packet Length', 'Bwd Packet Length Min', 'URG Flag Count', 'Fwd Packet Length Min', 'Destination Port']


## 9. Feature Distribution by Label (Violin Plots)

In [12]:
top_feats = label_corr.head(4).index.tolist()
sample_df2 = df.sample(min(30000, len(df)), random_state=42).copy()
sample_df2['label_binary'] = np.where(sample_df2['Label'] == 'BENIGN', 'BENIGN', 'ATTACK')

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, feat in zip(axes.flat, top_feats):
    sns.violinplot(data=sample_df2, x='label_binary', y=feat, ax=ax,
                  hue='label_binary',
                  palette={'BENIGN': '#2ecc71', 'ATTACK': '#e74c3c'},
                  inner='quartile', cut=0, legend=False)
    ax.set_title(f'Distribution of {feat}', fontsize=11)
    ax.set_xlabel('Traffic Type'); ax.set_ylabel(feat)
plt.suptitle('Top 4 Discriminative Features — Benign vs Attack', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/Feature Violin Plots.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Label Count across Protocols (Stacked Bar)

In [13]:
if 'Protocol' in df.columns:
    proto_label = df.groupby(['Protocol', 'Label']).size().unstack(fill_value=0)
    proto_label.plot(kind='bar', figsize=(12, 7), stacked=True,
                     colormap='tab20', edgecolor='black', lw=0.3)
    plt.title('Count of Label across various Protocols', fontsize=13)
    plt.xlabel('Protocol'); plt.ylabel('Flow Count')
    plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f'{IMAGES_DIR}/Count of Label across various Protocol.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Protocol column not available — skipping.')

Protocol column not available — skipping.


## 11. Byte Rate KDE Distributions (Benign vs Attack)

In [14]:
rate_feats = [f for f in ['Flow Bytes/s', 'Flow Packets/s'] if f in df.columns]

if rate_feats:
    sample_df3 = df.sample(min(20000, len(df)), random_state=42).copy()
    sample_df3['label_binary'] = np.where(sample_df3['Label'] == 'BENIGN', 'BENIGN', 'ATTACK')

    fig, axes = plt.subplots(1, len(rate_feats), figsize=(14, 5))
    if len(rate_feats) == 1:
        axes = [axes]
    for ax, feat in zip(axes, rate_feats):
        for lbl, color in [('BENIGN', '#2ecc71'), ('ATTACK', '#e74c3c')]:
            subset = sample_df3[sample_df3['label_binary'] == lbl][feat]
            subset = subset[subset.between(subset.quantile(0.01), subset.quantile(0.99))]
            sns.kdeplot(subset, ax=ax, label=lbl, color=color, fill=True, alpha=0.4)
        ax.set_title(f'KDE — {feat}', fontsize=12)
        ax.set_xlabel(feat); ax.set_ylabel('Density')
        ax.legend()
    plt.suptitle('Byte/Packet Rate Distributions: Benign vs Attack', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{IMAGES_DIR}/Byte Rate KDE Distributions.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Rate features not found in dataset.')

## 12. EDA Summary

| Insight | Finding |
|---------|----------|
| Class balance | Heavily imbalanced — BENIGN dominates |
| Flow duration | Attacks (DDoS, DoS) have distinct duration patterns |
| Flag patterns | SYN/FIN/RST flags differ significantly across attack types |
| Byte rates | Attack flows often show extreme byte/packet rates |
| Feature correlations | Many rate-based features are highly correlated → motivates PCA |
| Protocol | TCP dominates all traffic types |

**These insights directly justify modeling choices made in `DV_Project_Model.ipynb`.**